STEP 1 — Read Silver Data

In [0]:
silver_df = spark.read.table("project_catalog.silver.sales_clean")


In [0]:
%sql
show columns in project_catalog.silver.sales_clean

col_name
Product_ID
Sale_Date
Sales_Rep
Region
Sales_Amount
Quantity_Sold
Product_Category
Unit_Cost
Unit_Price
Customer_Type


### **CREATE DIMENSIONS**

DimProducts

get scd type full history

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimProducts (
  DimProductKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Product_ID STRING,
  Product_Category STRING,
  StartDate TIMESTAMP,
  EndDate TIMESTAMP,
  IsCurrent BOOLEAN
)
USING DELTA
CLUSTER BY (DimProductKey);

DimRegion

rest including region gets scd type 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimRegion (
  DimRegionKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Region STRING
)
USING DELTA
CLUSTER BY (DimRegionKey);


DimCustomers

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimCustomers(
  DimCustomerKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Customer_Type STRING
)
USING DELTA
CLUSTER BY (DimCustomerKey)

DimSalesRep

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimSalesRep(
  DimRepKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Sales_Rep STRING
)
USING DELTA
CLUSTER BY (DimRepKey)

DimChannel

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimChannel(
  DimChannelKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Sales_Channel STRING
)
USING DELTA
CLUSTER BY (DimChannelKey)

DimPayments

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimPayments(
  DimPaymentKey BIGINT GENERATED BY DEFAULT AS IDENTITY,
  Payment_Method STRING
)
USING DELTA
CLUSTER BY (DimPaymentKey)

DimDate

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.DimDate (
    DateKey INT NOT NULL,
    FullDate DATE NOT NULL,
    Year INT NOT NULL,
    Quarter INT NOT NULL,
    Month INT NOT NULL,
    MonthName STRING NOT NULL,
    DayOfMonth INT NOT NULL,
    DayOfWeek STRING NOT NULL,
    IsWeekend BOOLEAN NOT NULL
) 
USING DELTA 
CLUSTER BY (DateKey);

Step 2: The SCD Type 2 Python Code (For DimProducts)

In [0]:
# ================================
# SCD TYPE 2 IMPLEMENTATION
# DIM: PRODUCTS
# ================================

from pyspark.sql.functions import col

# STEP 1 — Read Silver data (clean layer)
silver_df = spark.read.table("project_catalog.silver.sales_clean")

# STEP 2 — Extract latest product view (Force 1 row per ID)
incoming_products = silver_df.select("Product_ID", "Product_Category") \
                             .dropDuplicates(["Product_ID"])

# Create temp view for SQL operations
incoming_products.createOrReplaceTempView("source_products")




# ================================
# STEP 3 — EXPIRE OLD RECORDS
# ================================
# If product exists AND category changed:
# → mark old record as NOT current
# → set EndDate

spark.sql("""
-- Move the IsCurrent check into the MATCHED section
MERGE INTO project_catalog.gold.DimProducts AS target
USING source_products AS source
ON target.Product_ID = source.Product_ID 
-- We only want to flip the status of the record that is currently active 
-- AND has a category that changed.
WHEN MATCHED 
    AND target.IsCurrent = true 
    AND target.Product_Category <> source.Product_Category 
THEN UPDATE SET
    target.EndDate = current_timestamp(),
    target.IsCurrent = false;
""")


# ================================
# STEP 4 — INSERT NEW RECORDS
# ================================
# Insert:
# 1. Completely new products
# 2. Updated products (after old version expired)


spark.sql("""
INSERT INTO project_catalog.gold.DimProducts
(Product_ID, Product_Category, StartDate, EndDate, IsCurrent)

SELECT 
    src.Product_ID,
    src.Product_Category,
    -- Using a historical date instead of current_timestamp()
    CAST('2010-01-01' AS TIMESTAMP) AS StartDate, 
    NULL AS EndDate,
    true AS IsCurrent

FROM source_products src

-- Try to find CURRENT version of product
LEFT JOIN project_catalog.gold.DimProducts tgt
    ON src.Product_ID = tgt.Product_ID 
    AND tgt.IsCurrent = true

-- Only insert if NO current version exists
WHERE tgt.Product_ID IS NULL
""")


print("✅ DimProducts updated using SCD Type 2 logic")

✅ DimProducts updated using SCD Type 2 logic


Step 3: The SCD Type 1 Python Code (For DimRegion)

In [0]:
# 1. Grab distinct regions from Silver
incoming_regions = silver_df.select("Region").distinct()
incoming_regions.createOrReplaceTempView("source_regions")

# 2. Upsert (SCD Type 1)
spark.sql("""
    MERGE INTO project_catalog.gold.DimRegion AS target
    USING source_regions AS source
    ON target.Region = source.Region
    WHEN NOT MATCHED THEN
      INSERT (Region) VALUES (source.Region)
""")

print("✅ DimRegion updated with SCD Type 1!")


✅ DimRegion updated with SCD Type 1!


DimCustomers

In [0]:
incoming_customers = silver_df.select('Customer_Type').distinct()

incoming_customers.createOrReplaceTempView("source_customers")


spark.sql("""
          MERGE INTO project_catalog.gold.DimCustomers AS target
          USING source_customers AS source
          ON source.Customer_Type = target.Customer_Type
          WHEN NOT MATCHED THEN 
          INSERT (Customer_Type) VALUES (source.Customer_Type)

          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

DimSalesRep

In [0]:
incoming_sales_rep = silver_df.select('Sales_Rep').distinct()

incoming_sales_rep.createOrReplaceTempView("source_sales_rep")


spark.sql("""
          MERGE INTO project_catalog.gold.DimSalesRep AS target
          USING source_sales_rep AS source
          ON source.Sales_Rep = target.Sales_Rep
          WHEN NOT MATCHED THEN 
          INSERT (Sales_Rep) VALUES (source.Sales_Rep)

          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

DimChannel

In [0]:
incoming_channel = silver_df.select('Sales_Channel').distinct()

incoming_channel.createOrReplaceTempView("source_Sales_Channel")


spark.sql("""
          MERGE INTO project_catalog.gold.DimChannel AS target
          USING source_Sales_Channel AS source
          ON source.Sales_Channel = target.Sales_Channel
          WHEN NOT MATCHED THEN 
          INSERT (Sales_Channel) VALUES (source.Sales_Channel)

          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

DimPayments

In [0]:
incoming_payments = silver_df.select('Payment_Method').distinct()

incoming_payments.createOrReplaceTempView("source_Payment_Method")


spark.sql("""
          MERGE INTO project_catalog.gold.DimPayments AS target
          USING source_Payment_Method AS source
          ON source.Payment_Method = target.Payment_Method
          WHEN NOT MATCHED THEN 
          INSERT (Payment_Method) VALUES (source.Payment_Method)

          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

DimDate

In [0]:
from pyspark.sql.functions import explode, sequence, to_date, col, year, quarter, month, date_format, dayofmonth, expr

# date range (Industry standard is usually 10 years back, 10 years forward)
begin_date = '2010-01-01'
end_date = '2030-12-31'

# Generate every single day in that range
df_dates = spark.sql(f"SELECT explode(sequence(to_date('{begin_date}'), to_date('{end_date}'), interval 1 day)) as FullDate")

# Add the professional attributes
dim_date = df_dates.select(
    (date_format(col("FullDate"), "yyyyMMdd").cast("int")).alias("DateKey"),
    col("FullDate"),
    year(col("FullDate")).alias("Year"),
    quarter(col("FullDate")).alias("Quarter"),
    month(col("FullDate")).alias("Month"),
    date_format(col("FullDate"), "MMMM").alias("MonthName"),
    dayofmonth(col("FullDate")).alias("DayOfMonth"),
    date_format(col("FullDate"), "EEEE").alias("DayOfWeek"),
    expr("CASE WHEN date_format(FullDate, 'EEEE') IN ('Sat', 'Sun') THEN true ELSE false END").alias("IsWeekend")
)

# Overwrite because this is a static reference table
dim_date.write.format("delta").mode("overwrite").saveAsTable("project_catalog.gold.DimDate")


### **CREATE FACT TABLE**

In [0]:
%sql
DROP TABLE IF EXISTS project_catalog.gold.FactSales;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_catalog.gold.FactSales (
  -- Natural Keys for Merging/Matching
  Product_ID STRING,    
  Sales_Rep STRING,     
  Sale_Date DATE,
  
  -- Measures
  Sales_Amount DECIMAL(18,2),
  Quantity_Sold INT,
  Unit_Cost DECIMAL(18,2),
  Unit_Price DECIMAL(18,2),
  Discount DECIMAL(18,2),
  
  -- Dimension Keys
  DimDateKey INT,
  DimProductKey BIGINT,
  DimRegionKey BIGINT,
  DimCustomerKey BIGINT,
  DimRepKey BIGINT,
  DimChannelKey BIGINT,
  DimPaymentKey BIGINT
)
USING DELTA
CLUSTER BY (Sale_Date);


In [0]:
# 1. Create a temp view of the silver data joined with ALL Gold Dimension Keys
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW incoming_fact_data AS
SELECT 
    F.Sales_Amount, F.Quantity_Sold, F.Unit_Cost, F.Unit_Price, F.Discount, F.Sale_Date,
    D.DateKey AS DimDateKey,
    P.DimProductKey,
    R.DimRegionKey,
    C.DimCustomerKey,
    SR.DimRepKey,
    CH.DimChannelKey,
    PY.DimPaymentKey,
    F.Product_ID,   
    F.Sales_Rep    
FROM project_catalog.silver.sales_clean AS F
-- 1. TIME SENSITIVE JOIN (SCD Type 2)
LEFT JOIN project_catalog.gold.DimProducts AS P
  ON F.Product_ID = P.Product_ID 
  AND F.Sale_Date >= CAST(P.StartDate AS DATE) -- Match date to date
  AND (F.Sale_Date < CAST(P.EndDate AS DATE) OR P.EndDate IS NULL)
-- 2. CALENDAR JOIN
LEFT JOIN project_catalog.gold.DimDate AS D 
  ON F.Sale_Date = D.FullDate
-- 3. STANDARD JOINS (SCD Type 1)
LEFT JOIN project_catalog.gold.DimRegion AS R ON F.Region = R.Region
LEFT JOIN project_catalog.gold.DimCustomers AS C ON F.Customer_Type = C.Customer_Type
LEFT JOIN project_catalog.gold.DimSalesRep AS SR ON F.Sales_Rep = SR.Sales_Rep
LEFT JOIN project_catalog.gold.DimChannel AS CH ON F.Sales_Channel = CH.Sales_Channel
LEFT JOIN project_catalog.gold.DimPayments AS PY ON F.Payment_Method = PY.Payment_Method
""")

# 2. MERGE into Fact Table
# We match on Product, Rep, and Date to ensure no duplicate transactions are inserted
spark.sql("""
MERGE INTO project_catalog.gold.FactSales AS target
USING incoming_fact_data AS source
ON target.Product_ID = source.Product_ID 
   AND target.Sales_Rep = source.Sales_Rep
   AND target.Sale_Date = source.Sale_Date
WHEN NOT MATCHED THEN 

  INSERT (Sales_Amount, Quantity_Sold, Unit_Cost, Unit_Price, Discount, Sale_Date, DimDateKey, 
  DimProductKey, DimRegionKey, DimCustomerKey, DimRepKey, DimChannelKey, DimPaymentKey)
  
  VALUES (source.Sales_Amount, source.Quantity_Sold, source.Unit_Cost, source.Unit_Price, source.Discount, 
  source.Sale_Date, source.DimDateKey, source.DimProductKey, source.DimRegionKey, 
  source.DimCustomerKey, source.DimRepKey, source.DimChannelKey, source.DimPaymentKey)
""")

print("FactSales successfully updated!")


✅ FactSales successfully updated!


### **GOLD VIEW**

In [0]:
%sql
CREATE OR REPLACE VIEW project_catalog.gold.v_Sales_Overview AS
SELECT 
    -- Measures
    F.Sales_Amount,
    F.Quantity_Sold,
    F.Unit_Price,
    F.Discount,
    
    -- Date Attributes (from DimDate)
    D.FullDate,
    D.Year,
    D.MonthName,
    D.DayOfWeek,
    
    -- Product Attributes (the correct version from SCD Type 2)
    P.Product_ID,
    P.Product_Category,
    
    -- Other Dimension Attributes
    R.Region,
    C.Customer_Type,
    SR.Sales_Rep,
    CH.Sales_Channel,
    PY.Payment_Method

FROM project_catalog.gold.FactSales AS F
LEFT JOIN project_catalog.gold.DimDate AS D ON F.DimDateKey = D.DateKey
LEFT JOIN project_catalog.gold.DimProducts AS P ON F.DimProductKey = P.DimProductKey
LEFT JOIN project_catalog.gold.DimRegion AS R ON F.DimRegionKey = R.DimRegionKey
LEFT JOIN project_catalog.gold.DimCustomers AS C ON F.DimCustomerKey = C.DimCustomerKey
LEFT JOIN project_catalog.gold.DimSalesRep AS SR ON F.DimRepKey = SR.DimRepKey
LEFT JOIN project_catalog.gold.DimChannel AS CH ON F.DimChannelKey = CH.DimChannelKey
LEFT JOIN project_catalog.gold.DimPayments AS PY ON F.DimPaymentKey = PY.DimPaymentKey;


In [0]:
%sql
SELECT * FROM project_catalog.gold.v_Sales_Overview LIMIT 20;

Sales_Amount,Quantity_Sold,Unit_Price,Discount,FullDate,Year,MonthName,DayOfWeek,Product_ID,Product_Category,Region,Customer_Type,Sales_Rep,Sales_Channel,Payment_Method
7154.95,27,1286.92,0.07,2023-02-18,2023,February,Saturday,1001,Clothing,West,Returning,David,Retail,Credit Card
5488.11,2,2904.06,0.15,2023-04-14,2023,April,Friday,1001,Clothing,East,New,David,Online,Bank Transfer
3167.09,25,1543.69,0.27,2023-04-27,2023,April,Thursday,1001,Clothing,East,New,Eve,Online,Credit Card
3793.91,47,5316.13,0.06,2023-05-10,2023,May,Wednesday,1001,Clothing,East,New,David,Online,Bank Transfer
5262.35,8,803.87,0.18,2023-05-11,2023,May,Thursday,1001,Clothing,East,New,Eve,Retail,Cash
2669.46,35,3244.76,0.05,2023-05-14,2023,May,Sunday,1001,Clothing,South,New,Eve,Online,Bank Transfer
1498.11,7,4576.50,0.30,2023-05-27,2023,May,Saturday,1001,Clothing,South,New,Alice,Retail,Cash
5879.35,20,2303.70,0.01,2023-08-03,2023,August,Thursday,1001,Clothing,West,Returning,Bob,Retail,Cash
9087.60,20,3563.97,0.25,2023-08-04,2023,August,Friday,1001,Clothing,West,Returning,David,Retail,Cash
8247.54,4,1871.22,0.12,2023-08-05,2023,August,Saturday,1001,Clothing,North,Returning,Bob,Online,Bank Transfer
